In [ ]:
# This file is for generating context_ids_chains that will be used for generating QAs

In [1]:
# Share the google drive
from google.colab import drive
drive.mount('/content/drive')

# Load the structured aviation manual
file_path = "/content/drive/MyDrive/Haystack/Dataset/manual.json"
import json
with open(file_path, "r", encoding="utf-8") as file:
  manual=json.load(file)

Mounted at /content/drive


In [2]:
titles = [context['title'] for context in manual]
contexts = [context['context'] for context in manual]

# Normalize the titles into the format of title chains
normalized_titles=[]
for title in titles:
    normalized_title = [t.strip() for t in title.split(">") if t.strip()]
    normalized_titles.append(normalized_title)

# Establish the title tree
# Define Node
class Node:
    def __init__(self, name, context_id = -1, parent = None, children = None):
        self.name = name
        self.context_id = context_id
        self.parent = parent
        self.children = children if children is not None else []
        if parent:
            parent.children.append(self)
    def add_child(self, child):
        self.children.append(child)
        child.parent = self
    def find_node(self, name):
        if self.name == name:
            return self
        for child in self.children:
            result = child.find_node(name)
            if result:
                return result
        return None
    def find_child(self, name):
        for child in self.children:
            if child.name == name:
                return child
        return None
    def clear(self):
        self.parent = None
        self.children = []
    def print_tree(self, level=0):
        print("   " * level + f"{self.name}: {self.context_id}")
        for child in self.children:
            child.print_tree(level + 1)
    def iterate(self):
        yield self
        for child in self.children:
            yield from child.iterate()
# Define add_path
def add_path(root, path, context_id):
    parent = root
    for i in range(len(path)):
        title = path[i]
        child = parent.find_child(title)
        if child:
            parent = child
        elif i != len(path)-1:
            child = Node(title, -1, parent)
            parent = child
        elif i == len(path)-1:
            child = Node(title, context_id, parent)
            parent = child
# Establish the title tree
book = Node("ARJ21")
for i in range(len(normalized_titles)):
  add_path(book, normalized_titles[i], i)

In [3]:
book.print_tree()

ARJ21: -1
   AIRCRAFT GENERAL: -1
      General Arrangement: 0
         Aircraft Geometric Dimension Limits: 1
         Antenna Locations: 2
         Ground Minimum Turning Radius: 3
         Ground Service Connections: 4
   AIR CONDITIONING  /PRESSURIZATION: 5
      Air Conditioning: -1
         System Description: -1
            General: 6
            Operating Logic: 7
               Integrated Air System Controllers: 8
            Flow Control: 9
            Air Conditioning Process: -1
               PACK: 10
               Emergency RAM Air: 11
               Mixing Manifold: 12
               Recirculation Fan: 13
               Trim Air: 14
            Temperature Control: -1
               Cabin: 15
               Cargo Compartment: 16
            Ventilation: -1
               Cargo Compartment Ventilation: 17
               Electronic Equipment Bay (E/E) Ventilation: 18
         Controls and Indications: -1
            AIR CONDITION Control Panel: 19
            ECS Synoptic

In [ ]:
# Linear Descent Sampling
LDS_chains = []
def LDS(node, path, LDS_chains, min_effective_length = 4):
  # If a node has no child, it means this sample has reached the bottom.
  # If this sample's path is long enough, then append this path to the chains
  if node.children == []:
    if len(path) >= min_effective_length:
        LDS_chains.append(path)

  # Otherwise, a node has children.
  # Iterate all the children to create samples.
  else:
    for child in node.children:
      # If a child has context_id = -1, then it means this title has no context.
      # If a title has no context, then it is excluded in the path.
      if child.context_id == -1:
        LDS(child, path, LDS_chains, min_effective_length)
      else:
        LDS(child, path + [child.context_id], LDS_chains, min_effective_length)

LDS(node = book, path = [], LDS_chains = LDS_chains, min_effective_length = 4)

In [ ]:
# Special note:
# The first id for each chain in LDS_chains always refers to the immediate children of the book node.
# For the manual I used, immediate children of the book node are all tables of contents, which are useless for generating questions.
# Therefore, the first id in each chain in LDS_chains will be deleted.
LDS_chains = [chain[1:] for chain in LDS_chains]

In [ ]:
# Exhaustive Sibling Sampling
ESS_chains = []
def ESS(node, ESS_chains, min_effective_length = 3):
  # Iterate all the nodes to create samples.
  for node in node.iterate():
    ESS_OneNode(node, ESS_chains, min_effective_length)

def ESS_OneNode(node, ESS_chains, min_effective_length):
  # If a node has children, then try to make a sample.
  # Sample should delete the title that has no context, which refers to the node that has context_id = -1.
  # If this sample's path is long enough, then append this path to the chains
  if node.children != []:
    if node.context_id == -1:
      path = [child.context_id for child in node.children if child.context_id != -1]
    else:
      path = [node.context_id] + [child.context_id for child in node.children if child.context_id != -1]
    if len(path) >= min_effective_length:
        ESS_chains.append(path)

ESS(node = book, ESS_chains = ESS_chains, min_effective_length = 3)

In [ ]:
# Special note:
# The first chain in ESS_chain is all tables of contents. So The first chain is deleted.
ESS_chains = ESS_chains[1:]

In [ ]:
# Random Node Sampling:
import random
RNS_chains = []
def RNS(root, RNS_chains, num_samples = 170, effective_length = 3):
  # Only save id that have context and are not tables of contents
  useful_ids = [node.context_id for node in root.iterate() if (node not in root.children and node.context_id != -1)]

  # Randomly select three ids from the useful_ids to create a sample
  count = 0
  while count < num_samples:
    sample = random.sample(useful_ids, effective_length)
    # Make sure this sample is not repeated.
    if sample not in RNS_chains:
      RNS_chains.append(sample)
      count += 1

RNS(root = book, RNS_chains = RNS_chains, num_samples = 170, effective_length = 3)

In [ ]:
# Save LDS_chains, ESS_chains and RNS_chains for later use.
context_ids_chains = {"LDS": LDS_chains, "ESS": ESS_chains, "RNS": RNS_chains}
json_path = "/content/drive/MyDrive/Haystack/Dataset/manual_context_ids_chains.json"
with open(json_path, 'w') as f:
    json.dump(context_ids_chains, f, indent=4)